In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import ast
from typing import Dict, Tuple, List

class MTRSPSolver:
    """Multi-Tugboat Routing and Scheduling Problem Solver using Gurobi"""
    
    def __init__(self):
        self.model = None
        self.instance_data = None
        self.variables = {}
        self.solution = {}
        
    def load_data(self, data_path: str) -> dict:
        import json
            
        with open(data_path, 'r') as f:
            instance = json.load(f)
        
        tasks = instance['tasks']
        tugboats = instance['tugboats']
        num_tasks = len(tasks)
        num_tugboats = len(tugboats)
        
        return {
            "num_tasks": num_tasks,
            "num_tugboats": num_tugboats,
            "task_max_tugs": [tasks[str(s)]['num_tugs_needed'] for s in range(1, num_tasks+1)],
            "task_min_horsepower": [tasks[str(s)]['min_horsepower'] for s in range(1, num_tasks+1)],
            "task_time_window_lower": [tasks[str(s)]['time_window'][0] for s in range(1, num_tasks+1)],
            "task_time_window_upper": [tasks[str(s)]['time_window'][1] for s in range(1, num_tasks+1)],
            "task_service_time": [tasks[str(s)]['service_time'] for s in range(1, num_tasks+1)],
            "tugboat_horsepower": [tugboats[str(k)]['horsepower'] for k in range(1, num_tugboats+1)],
            "tugboat_fuel_capacity": [tugboats[str(k)]['fuel_capacity'] for k in range(1, num_tugboats+1)],
            "tugboat_alpha": [tugboats[str(k)]['alpha'] for k in range(1, num_tugboats+1)],
            "tugboat_beta": [tugboats[str(k)]['beta'] for k in range(1, num_tugboats+1)],
            "time_matrix": instance['time_matrix'],
            "big_M": instance['metadata']['M'],
            "planning_horizon": instance['metadata']['T_max'],
            "penalty_weight": instance['metadata']['W']
        }
        
    def build_model(self, instance_data: dict):
        """Build the Gurobi optimization model for MTRSP"""
        self.instance_data = instance_data
        
        # Extract parameters
        num_tasks = instance_data['num_tasks']
        num_tugboats = instance_data['num_tugboats']
        
        # Task parameters (convert to lists if numpy arrays)
        task_max_tugs = list(instance_data['task_max_tugs'])
        task_min_horsepower = list(instance_data['task_min_horsepower'])
        task_time_window_lower = list(instance_data['task_time_window_lower'])
        task_time_window_upper = list(instance_data['task_time_window_upper'])
        task_service_time = list(instance_data['task_service_time'])
        
        # Tugboat parameters
        tugboat_horsepower = list(instance_data['tugboat_horsepower'])
        tugboat_fuel_capacity = list(instance_data['tugboat_fuel_capacity'])
        tugboat_alpha = list(instance_data['tugboat_alpha'])
        tugboat_beta = list(instance_data['tugboat_beta'])
        
        # Other parameters
        time_matrix = instance_data['time_matrix']
        big_M = instance_data['big_M']
        planning_horizon = instance_data['planning_horizon']
        penalty_weight = instance_data['penalty_weight']
        
        # Create model
        self.model = gp.Model("MTRSP")
        
        # Sets
        tasks = range(1, num_tasks + 1)  # 1 to n
        tugboats = range(1, num_tugboats + 1)  # 1 to K
        depot_start = 0
        depot_end = num_tasks + 1
        all_nodes = [depot_start] + list(tasks) + [depot_end]
        
        # ==================== Decision Variables ====================
        
        # x[i,j,k]: Binary variable for tugboat k traveling from node i to node j
        x = {}
        for k in tugboats:
            # From depot to tasks
            for j in tasks:
                x[depot_start, j, k] = self.model.addVar(vtype=GRB.BINARY, 
                                                          name=f'x_{depot_start}_{j}_{k}')
            # Between tasks
            for i in tasks:
                for j in tasks:
                    if i != j:
                        x[i, j, k] = self.model.addVar(vtype=GRB.BINARY, 
                                                        name=f'x_{i}_{j}_{k}')
                # From tasks to depot
                x[i, depot_end, k] = self.model.addVar(vtype=GRB.BINARY, 
                                                        name=f'x_{i}_{depot_end}_{k}')
        
        # y[s,k]: Binary variable for tugboat k serving task s
        y = {}
        for s in tasks:
            for k in tugboats:
                y[s, k] = self.model.addVar(vtype=GRB.BINARY, name=f'y_{s}_{k}')
        
        # z[s]: Binary variable indicating if task s is selected
        z = {}
        for s in tasks:
            z[s] = self.model.addVar(vtype=GRB.BINARY, name=f'z_{s}')
        
        # tau[s]: Service start time for task s
        tau = {}
        for s in tasks:
            tau[s] = self.model.addVar(vtype=GRB.CONTINUOUS, lb=0, 
                                        name=f'tau_{s}')
        
        self.model.update()
        
        # Store variables
        self.variables = {'x': x, 'y': y, 'z': z, 'tau': tau}
        
        # ==================== Objective Function ====================
        
        # Service fuel consumption
        Z_service = gp.quicksum(
            y[s, k] * tugboat_alpha[k-1] * tugboat_horsepower[k-1] * task_service_time[s-1]
            for s in tasks for k in tugboats
        )
        
        # Transit fuel consumption
        Z_transit = gp.LinExpr()
        for k in tugboats:
            # From depot to tasks
            for j in tasks:
                key = f'{depot_start}_{j}'
                if key in time_matrix:
                    t_ij = time_matrix[key]
                    Z_transit += x[depot_start, j, k] * tugboat_beta[k-1] * tugboat_horsepower[k-1] * t_ij
            
            # Between tasks
            for i in tasks:
                for j in tasks:
                    if i != j:
                        key = f'{i}_{j}'
                        if key in time_matrix:
                            t_ij = time_matrix[key]
                            Z_transit += x[i, j, k] * tugboat_beta[k-1] * tugboat_horsepower[k-1] * t_ij
                
                # From tasks to depot
                key = f'{i}_{depot_end}'
                if key in time_matrix:
                    t_ij = time_matrix[key]
                    Z_transit += x[i, depot_end, k] * tugboat_beta[k-1] * tugboat_horsepower[k-1] * t_ij
        
        # Total fuel consumption
        Z_fuel = Z_service + Z_transit
        
        # Penalty for uncompleted tasks
        Z_penalty = gp.quicksum(1 - z[s] for s in tasks)
        
        # Total objective
        objective = Z_fuel + penalty_weight * Z_penalty
        
        self.model.setObjective(objective, GRB.MINIMIZE)
        
        # ==================== Constraints ====================
        
        # (C1) Tugboat quantity constraint
        for s in tasks:
            self.model.addConstr(
                gp.quicksum(y[s, k] for k in tugboats) <= task_max_tugs[s-1] * z[s],
                name=f'C1_tugboat_quantity_{s}'
            )
        
        # (C2) Horsepower requirement constraint
        for s in tasks:
            self.model.addConstr(
                gp.quicksum(y[s, k] * tugboat_horsepower[k-1] for k in tugboats) 
                >= task_min_horsepower[s-1] * z[s],
                name=f'C2_horsepower_{s}'
            )
        
        # (C3) Each tugboat departs from depot at most once
        for k in tugboats:
            self.model.addConstr(
                gp.quicksum(x[depot_start, j, k] for j in tasks) <= 1,
                name=f'C3_depart_depot_{k}'
            )
        
        # (C4) Each tugboat returns to depot at most once
        for k in tugboats:
            self.model.addConstr(
                gp.quicksum(x[i, depot_end, k] for i in tasks) <= 1,
                name=f'C4_return_depot_{k}'
            )
        
        # (C5) Flow conservation constraint
        for j in tasks:
            for k in tugboats:
                inflow = x[depot_start, j, k] + gp.quicksum(x[i, j, k] for i in tasks if i != j)
                outflow = gp.quicksum(x[j, i, k] for i in tasks if i != j) + x[j, depot_end, k]
                self.model.addConstr(inflow == outflow, name=f'C5_flow_conservation_{j}_{k}')
        
        # (C6) Linkage between service indicator and arc variables
        for s in tasks:
            for k in tugboats:
                inflow_to_s = x[depot_start, s, k] + gp.quicksum(x[i, s, k] for i in tasks if i != s)
                self.model.addConstr(y[s, k] == inflow_to_s, name=f'C6_service_linkage_{s}_{k}')
        
        # (C7) Service time must be within time window
        for s in tasks:
            self.model.addConstr(
                tau[s] >= task_time_window_lower[s-1] * z[s],
                name=f'C7_time_window_lower_{s}'
            )
            self.model.addConstr(
                tau[s] <= task_time_window_upper[s-1] + big_M * (1 - z[s]),
                name=f'C7_time_window_upper_{s}'
            )
        
        # (C8) Service must be completed within planning horizon
        for s in tasks:
            self.model.addConstr(
                tau[s] + task_service_time[s-1] <= planning_horizon + big_M * (1 - z[s]),
                name=f'C8_planning_horizon_{s}'
            )
        
        # (C9) Time constraint for departure from depot
        for j in tasks:
            for k in tugboats:
                key = f'{depot_start}_{j}'
                if key in time_matrix:
                    t_0j = time_matrix[key]
                    self.model.addConstr(
                        tau[j] >= t_0j - big_M * (1 - x[depot_start, j, k]),
                        name=f'C9_depot_time_{j}_{k}'
                    )
        
        # (C10) Time propagation constraint between tasks
        for i in tasks:
            for j in tasks:
                if i != j:
                    for k in tugboats:
                        key = f'{i}_{j}'
                        if key in time_matrix:
                            t_ij = time_matrix[key]
                            self.model.addConstr(
                                tau[j] >= tau[i] + task_service_time[i-1] + t_ij 
                                - big_M * (1 - x[i, j, k]),
                                name=f'C10_time_propagation_{i}_{j}_{k}'
                            )
        
        # (C11) Fuel capacity constraint
        for k in tugboats:
            service_fuel = gp.quicksum(
                y[s, k] * tugboat_alpha[k-1] * tugboat_horsepower[k-1] * task_service_time[s-1]
                for s in tasks
            )
            
            transit_fuel = gp.LinExpr()
            # From depot
            for j in tasks:
                key = f'{depot_start}_{j}'
                if key in time_matrix:
                    transit_fuel += x[depot_start, j, k] * tugboat_beta[k-1] * tugboat_horsepower[k-1] * time_matrix[key]
            
            # Between tasks
            for i in tasks:
                for j in tasks:
                    if i != j:
                        key = f'{i}_{j}'
                        if key in time_matrix:
                            transit_fuel += x[i, j, k] * tugboat_beta[k-1] * tugboat_horsepower[k-1] * time_matrix[key]
                
                # To depot
                key = f'{i}_{depot_end}'
                if key in time_matrix:
                    transit_fuel += x[i, depot_end, k] * tugboat_beta[k-1] * tugboat_horsepower[k-1] * time_matrix[key]
            
            self.model.addConstr(
                service_fuel + transit_fuel <= tugboat_fuel_capacity[k-1],
                name=f'C11_fuel_capacity_{k}'
            )
        
        self.model.update()
        print("Model built successfully!")
        
    def solve(self, time_limit=3600, mip_gap=0.01, log_file=None):
        """Solve the optimization model"""
        if self.model is None:
            raise ValueError("Model has not been built yet. Call build_model() first.")
        
        # Set parameters
        self.model.Params.TimeLimit = time_limit
        self.model.Params.MIPGap = mip_gap
        
        if log_file:
            self.model.Params.LogFile = log_file
        
        # Solve
        print("Starting optimization...")
        self.model.optimize()
        
        # Extract solution
        if self.model.Status == GRB.OPTIMAL or self.model.Status == GRB.TIME_LIMIT:
            self.extract_solution()
        else:
            print(f"Optimization ended with status {self.model.Status}")
    
    def extract_solution(self):
        """Extract solution from the model - WITH DETAILED DIAGNOSIS"""
        if self.model.SolCount == 0:
            print("No solution found.")
            return
        
        num_tasks = self.instance_data['num_tasks']
        num_tugboats = self.instance_data['num_tugboats']
        
        tasks = range(1, num_tasks + 1)
        tugboats = range(1, num_tugboats + 1)
        
        x = self.variables['x']
        y = self.variables['y']
        z = self.variables['z']
        tau = self.variables['tau']
        
        # Extract task parameters
        task_max_tugs = self.instance_data['task_max_tugs']
        task_service_time = self.instance_data['task_service_time']
        tugboat_alpha = self.instance_data['tugboat_alpha']
        tugboat_beta = self.instance_data['tugboat_beta']
        tugboat_horsepower = self.instance_data['tugboat_horsepower']
        time_matrix = self.instance_data['time_matrix']
        depot_end = num_tasks + 1
        
        self.solution = {
            'objective_value': self.model.ObjVal,
            'mip_gap': self.model.MIPGap if self.model.Status == GRB.TIME_LIMIT else 0,
            'selected_tasks': [],
            'task_start_times': {},
            'task_assignments': {},
            'tugboat_routes': {}
        }
        
        # ========== DETAILED DIAGNOSIS ==========
        print("\n" + "="*70)
        print("DETAILED SOLUTION DIAGNOSIS")
        print("="*70)
        
        # Selected tasks and assignments
        for s in tasks:
            if z[s].X > 0.5:
                self.solution['selected_tasks'].append(s)
                self.solution['task_start_times'][s] = tau[s].X
                
                # Task assignments
                assigned_tugboats = []
                for k in tugboats:
                    if y[s, k].X > 0.5:
                        assigned_tugboats.append(k)
                self.solution['task_assignments'][s] = assigned_tugboats
        
        # Tugboat routes
        depot_start = 0
        
        for k in tugboats:
            route = []
            current = depot_start
            
            while True:
                found_next = False
                # Check all possible next nodes
                if current == depot_start:
                    for j in tasks:
                        if (current, j, k) in x and x[current, j, k].X > 0.5:
                            route.append(j)
                            current = j
                            found_next = True
                            break
                elif current in tasks:
                    # Check next task
                    for j in tasks:
                        if current != j and (current, j, k) in x and x[current, j, k].X > 0.5:
                            route.append(j)
                            current = j
                            found_next = True
                            break
                    # Check return to depot
                    if not found_next and (current, depot_end, k) in x and x[current, depot_end, k].X > 0.5:
                        found_next = True
                        break
                
                if not found_next:
                    break
            
            if route:
                self.solution['tugboat_routes'][k] = route
        
        # ========== CALCULATE DETAILED FUEL AND PENALTY ==========
        total_service_fuel = 0.0
        total_transit_fuel = 0.0
        
        # Service fuel - calculate for each (task, tugboat) pair
        print("\n服务燃料明细:")
        task_service_fuel = {}
        for s in self.solution['selected_tasks']:
            print(f"\nTask {s} (需要{task_max_tugs[s-1]}艘, 服务时间{task_service_time[s-1]:.2f}h):")
            task_fuel = 0.0
            for k in self.solution['task_assignments'][s]:
                service_fuel = (tugboat_alpha[k-1] * 
                               tugboat_horsepower[k-1] * 
                               task_service_time[s-1])
                task_fuel += service_fuel
                total_service_fuel += service_fuel
                print(f"  拖船 {k} (HP={tugboat_horsepower[k-1]:.0f}, α={tugboat_alpha[k-1]:.3f}): {service_fuel:.2f} kg")
            task_service_fuel[s] = task_fuel
            print(f"  任务总服务燃料: {task_fuel:.2f} kg")
        
        # Transit fuel - calculate for each tugboat route
        print("\n行驶燃料明细:")
        for k in sorted(self.solution['tugboat_routes'].keys()):
            route = self.solution['tugboat_routes'][k]
            print(f"\n拖船 {k} (HP={tugboat_horsepower[k-1]:.0f}, β={tugboat_beta[k-1]:.3f}):")
            print(f"  路径: Depot → {' → '.join(map(str, route))} → Depot")
            
            tug_transit_fuel = 0.0
            current = 0
            
            for next_node in route:
                key = f'{current}_{next_node}'
                if key in time_matrix:
                    t_ij = time_matrix[key]
                    fuel = tugboat_beta[k-1] * tugboat_horsepower[k-1] * t_ij
                    tug_transit_fuel += fuel
                    print(f"    {current} → {next_node}: t={t_ij:.4f}h, 燃料={fuel:.2f}kg")
                current = next_node
            
            # Return to depot
            key = f'{current}_{depot_end}'
            if key in time_matrix:
                t_ij = time_matrix[key]
                fuel = tugboat_beta[k-1] * tugboat_horsepower[k-1] * t_ij
                tug_transit_fuel += fuel
                print(f"    {current} → {depot_end}: t={t_ij:.4f}h, 燃料={fuel:.2f}kg")
            
            total_transit_fuel += tug_transit_fuel
            print(f"  拖船行驶总燃料: {tug_transit_fuel:.2f} kg")
        
        # Penalty calculation
        num_completed = len(self.solution['selected_tasks'])
        num_uncompleted = num_tasks - num_completed
        penalty_value = self.instance_data['penalty_weight'] * num_uncompleted
        
        print("\n" + "="*70)
        print("目标函数分解:")
        print("="*70)
        print(f"服务燃料 (Z_service):     {total_service_fuel:12.2f} kg")
        print(f"行驶燃料 (Z_transit):     {total_transit_fuel:12.2f} kg")
        print(f"总燃料 (Z_fuel):         {total_service_fuel + total_transit_fuel:12.2f} kg")
        print(f"未完成任务数:              {num_uncompleted:12d} 个")
        print(f"惩罚 (W × penalty):       {penalty_value:12.2f}")
        print(f"{'='*70}")
        print(f"总目标值:                 {total_service_fuel + total_transit_fuel + penalty_value:12.2f}")
        print(f"Gurobi报告目标值:         {self.model.ObjVal:12.2f}")
        print(f"差异:                     {abs(self.model.ObjVal - (total_service_fuel + total_transit_fuel + penalty_value)):12.2f}")
        print("="*70)
        
        print(f"\nNumber of Completed Tasks: {num_completed} / {num_tasks}")
        print(f"Number of Active Tugboats: {len(self.solution['tugboat_routes'])}")
        print(f"MIP Gap: {self.solution['mip_gap']:.2%}")
        
    def print_detailed_solution(self):
        """Print detailed solution information"""
        if not self.solution:
            print("No solution available.")
            return
        
        print("\n" + "="*60)
        print("DETAILED SOLUTION")
        print("="*60)
        
        # Selected tasks
        print(f"\nCompleted Tasks ({len(self.solution['selected_tasks'])}):")
        for s in sorted(self.solution['selected_tasks']):
            start_time = self.solution['task_start_times'][s]
            assigned_tugs = self.solution['task_assignments'][s]
            print(f"  Task {s}: Start time = {start_time:.2f}h, Tugboats = {assigned_tugs}")
        
        # Tugboat routes
        print(f"\nTugboat Routes ({len(self.solution['tugboat_routes'])}):")
        for k in sorted(self.solution['tugboat_routes'].keys()):
            route = self.solution['tugboat_routes'][k]
            route_str = ' -> '.join([str(node) for node in route])
            print(f"  Tugboat {k}: Depot -> {route_str} -> Depot")
    
    def export_solution(self, output_path: str):
        """Export solution to a text file"""
        if not self.solution:
            print("No solution to export.")
            return
        
        with open(output_path, 'w') as f:
            f.write("="*60 + "\n")
            f.write("MTRSP SOLUTION\n")
            f.write("="*60 + "\n\n")
            
            f.write(f"Objective Value: {self.solution['objective_value']:.2f}\n")
            f.write(f"Completed Tasks: {len(self.solution['selected_tasks'])}\n")
            f.write(f"Active Tugboats: {len(self.solution['tugboat_routes'])}\n")
            f.write(f"MIP Gap: {self.solution['mip_gap']:.2%}\n\n")
            
            f.write("Completed Tasks:\n")
            for s in sorted(self.solution['selected_tasks']):
                start_time = self.solution['task_start_times'][s]
                assigned_tugs = self.solution['task_assignments'][s]
                f.write(f"  Task {s}: Start={start_time:.2f}h, Tugboats={assigned_tugs}\n")
            
            f.write("\nTugboat Routes:\n")
            for k in sorted(self.solution['tugboat_routes'].keys()):
                route = self.solution['tugboat_routes'][k]
                route_str = ' -> '.join([str(node) for node in route])
                f.write(f"  Tugboat {k}: Depot -> {route_str} -> Depot\n")
        
        print(f"Solution exported to {output_path}")




In [3]:
# Example usage
if __name__ == "__main__":
    # Create solver instance
    solver = MTRSPSolver()
    
    # Load data from file
    data_file = r"C:\Users\gongh\拖船项目求解/MACE/output/mtrsp/data/test_data\X_10_3_v1.json"
    instance_data = solver.load_data(data_file)
    
    # Build model
    solver.build_model(instance_data)
    
    # Solve
    solver.solve(time_limit=1800, mip_gap=0.01, log_file="gurobi.log")
    
    # Print solution
    solver.print_detailed_solution()
    
    # Export solution
    solver.export_solution("gurobi_solution.txt")

Set parameter LicenseID to value 2613782
Model built successfully!
Set parameter TimeLimit to value 1800
Set parameter MIPGap to value 0.01
Set parameter LogFile to value "gurobi.log"
Starting optimization...
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i5-12450H, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 419 rows, 380 columns and 2360 nonzeros
Model fingerprint: 0x72ac12d4
Variable types: 10 continuous, 370 integer (370 binary)
Coefficient statistics:
  Matrix range     [5e-01, 2e+04]
  Objective range  [1e+02, 1e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+04]
Found heuristic solution: objective 100000.00000
Presolve removed 357 rows and 224 columns
Presolve time: 0.04s
Presolved: 62 rows, 156 columns, 581 nonzeros
Variable types: 0 continuous, 156 integer (156 binary)
Found heuristic solution: 